# crdt-merge v0.4.0 — A100 Validation
**The Trust Release**: Provenance tracking, `@verified_merge` decorator, streaming optimization

Tests v0.4.0-specific features + full system sanity check on NVIDIA A100.

In [ ]:
!pip install crdt-merge==0.4.0 psutil -q

import crdt_merge, sys, os, time, json, tracemalloc, gc
print(f'crdt-merge version: {crdt_merge.__version__}')
print(f'Python: {sys.version.split()[0]}')

try:
    import torch
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
except ImportError:
    print('torch not available (CPU-only)')

import psutil
print(f'RAM: {psutil.virtual_memory().total / 1e9:.1f} GB')
print(f'CPUs: {os.cpu_count()}')

## Utilities

In [ ]:
import numpy as np

results = []

def record(suite, test, value, unit, passed=True):
    results.append({'suite': suite, 'test': test, 'value': value, 'unit': unit, 'passed': passed})
    status = '' if passed else ''
    print(f'  {status} {test}: {value} {unit}')

def measure_memory(fn):
    gc.collect()
    tracemalloc.start()
    result = fn()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, peak / 1e6  # MB

def gen_records(n, id_start=0):
    return [{'id': i, 'value': int(np.random.randint(0, 1000)),
             'ts': float(i + np.random.random()),
             'name': f'item_{i}'} for i in range(id_start, id_start + n)]

print('Utilities loaded ')

## Suite 1: Provenance Tracking
Tests `merge_with_provenance()` and `export_provenance()` at scale.

In [ ]:
from crdt_merge import merge_with_provenance, export_provenance, MergeSchema, LWW

print('=' * 70)
print('SUITE 1: PROVENANCE')
print('=' * 70)

SUITE = 'provenance'

# --- Basic provenance ---
print('\n--- Basic Provenance ---')
a = [{'id': 1, 'name': 'Alice', 'ts': 1.0}, {'id': 2, 'name': 'Bob', 'ts': 2.0}]
b = [{'id': 2, 'name': 'Bobby', 'ts': 3.0}, {'id': 3, 'name': 'Charlie', 'ts': 1.0}]
merged, prov = merge_with_provenance(a, b, key='id', timestamp_col='ts')
record(SUITE, 'basic_merge_rows', len(merged), 'rows', len(merged) == 3)
record(SUITE, 'total_rows', prov.total_rows, 'rows', prov.total_rows == 3)
record(SUITE, 'merged_rows', prov.merged_rows, 'rows', prov.merged_rows == 1)
record(SUITE, 'unique_a_rows', prov.unique_a_rows, 'rows', prov.unique_a_rows == 1)
record(SUITE, 'unique_b_rows', prov.unique_b_rows, 'rows', prov.unique_b_rows == 1)
record(SUITE, 'total_conflicts', prov.total_conflicts, 'conflicts', prov.total_conflicts >= 0)

# --- Export formats ---
print('\n--- Export Formats ---')
json_out = export_provenance(prov, format='json')
parsed = json.loads(json_out)
record(SUITE, 'json_export', len(json_out), 'bytes', len(json_out) > 0)
record(SUITE, 'json_parseable', True, '', True)

csv_out = export_provenance(prov, format='csv')
record(SUITE, 'csv_export', len(csv_out), 'bytes', len(csv_out) > 0)

# --- Summary ---
print('\n--- Summary ---')
summary = prov.summary()
record(SUITE, 'summary_available', len(summary), 'chars', len(summary) > 0)
print(f'  Summary: {summary[:200]}')

# --- To dict ---
d = prov.to_dict()
record(SUITE, 'to_dict', len(str(d)), 'chars', isinstance(d, dict))

# --- Schema + provenance ---
print('\n--- Schema + Provenance ---')
schema = MergeSchema(default=LWW(), name=LWW())
a2 = [{'id': i, 'name': f'a_{i}', 'score': i*10, 'ts': float(i)} for i in range(100)]
b2 = [{'id': i, 'name': f'b_{i}', 'score': i*20, 'ts': float(i+100)} for i in range(50, 150)]
merged2, prov2 = merge_with_provenance(a2, b2, key='id', schema=schema, timestamp_col='ts')
record(SUITE, 'schema_provenance_rows', len(merged2), 'rows', len(merged2) == 150)
record(SUITE, 'schema_merged_count', prov2.merged_rows, 'rows', prov2.merged_rows == 50)
record(SUITE, 'schema_duration_ms', round(prov2.duration_ms, 2), 'ms', prov2.duration_ms >= 0)

# --- Scale test ---
print('\n--- Scale: 10K overlapping records ---')
a3 = gen_records(10000, 0)
b3 = gen_records(10000, 5000)
t0 = time.time()
merged3, prov3 = merge_with_provenance(a3, b3, key='id', timestamp_col='ts')
elapsed = (time.time() - t0) * 1000
record(SUITE, 'scale_10k_rows', len(merged3), 'rows', len(merged3) == 15000)
record(SUITE, 'scale_10k_time_ms', round(elapsed, 1), 'ms')
record(SUITE, 'scale_10k_conflicts', prov3.total_conflicts, 'conflicts')

print(f'\nProvenance suite: {sum(1 for r in results if r["suite"]==SUITE and r["passed"])}/{sum(1 for r in results if r["suite"]==SUITE)} passed')

## Suite 2: Verified Merge (`@verified_merge`)
Tests the trust layer — CRDT property verification.

In [ ]:
from crdt_merge import merge, verified_merge
from crdt_merge.verify import verify_crdt, verify_convergence

print('=' * 70)
print('SUITE 2: VERIFIED MERGE')
print('=' * 70)

SUITE = 'verify'

# gen_fn and merge_fn for verify APIs
def gen_fn():
    n = np.random.randint(5, 15)
    ids = np.random.choice(range(30), size=n, replace=False).tolist()
    return [{'id': i, 'val': int(np.random.randint(0, 100)),
             'ts': float(np.random.random() * 1000)} for i in ids]

def merge_fn(a, b):
    return merge(a, b, key='id', timestamp_col='ts')

def eq_fn(a, b):
    return sorted(a, key=lambda r: r['id']) == sorted(b, key=lambda r: r['id'])

# --- Full CRDT verification ---
print('\n--- Full CRDT Verification (100 trials) ---')
t0 = time.time()
res = verify_crdt(merge_fn, gen_fn, trials=100, eq_fn=eq_fn)
elapsed = (time.time() - t0) * 1000
record(SUITE, 'crdt_verification_passed', res.passed, '', res.passed)
record(SUITE, 'commutativity', str(res.commutativity), '', 'PASS' in str(res.commutativity))
record(SUITE, 'associativity', str(res.associativity), '', 'PASS' in str(res.associativity))
record(SUITE, 'idempotency', str(res.idempotency), '', 'PASS' in str(res.idempotency))
record(SUITE, 'convergence', str(res.convergence), '', 'PASS' in str(res.convergence))
record(SUITE, 'total_trials', res.total_trials, 'trials')
record(SUITE, 'total_duration_ms', round(res.total_duration_ms, 1), 'ms')

# --- Convergence at scale ---
print('\n--- Convergence (5 replicas, 50 trials) ---')
t0 = time.time()
res2 = verify_convergence(merge_fn, gen_fn, trials=50, num_replicas=5, eq_fn=eq_fn)
elapsed2 = (time.time() - t0) * 1000
record(SUITE, 'convergence_50trials', res2.passed, '', res2.passed)
record(SUITE, 'convergence_failures', res2.failures, 'failures', res2.failures == 0)
record(SUITE, 'convergence_time_ms', round(elapsed2, 1), 'ms')

# --- @verified_merge decorator ---
print('\n--- @verified_merge Decorator ---')

@verified_merge(gen_fn=gen_fn, trials=50, eq_fn=eq_fn, on_fail='raise')
def safe_merge(a, b):
    return merge(a, b, key='id', timestamp_col='ts')

# Should work without error
test_a = gen_fn()
test_b = gen_fn()
result = safe_merge(test_a, test_b)
record(SUITE, 'decorator_works', len(result), 'rows', len(result) > 0)

# --- Summary report ---
print('\n--- Verification Summary ---')
print(res.summary())

print(f'\nVerify suite: {sum(1 for r in results if r["suite"]==SUITE and r["passed"])}/{sum(1 for r in results if r["suite"]==SUITE)} passed')

## Suite 3: Streaming Merge
Tests `merge_sorted_stream()` and `merge_stream()` with memory tracking.

In [ ]:
from crdt_merge import merge_sorted_stream, merge_stream, StreamStats

print('=' * 70)
print('SUITE 3: STREAMING')
print('=' * 70)

SUITE = 'streaming'

# --- merge_sorted_stream basic ---
print('\n--- merge_sorted_stream (basic) ---')
a = gen_records(1000, 0)  # ids 0-999
b = gen_records(1000, 500)  # ids 500-1499
stats = StreamStats()
batches = list(merge_sorted_stream(iter(a), iter(b), key='id', timestamp_col='ts', stats=stats))
total = sum(len(batch) for batch in batches)
record(SUITE, 'sorted_basic_rows', total, 'rows', total == 1500)
record(SUITE, 'sorted_batches', len(batches), 'batches', len(batches) >= 1)
record(SUITE, 'sorted_stats_rows_processed', stats.rows_processed, 'rows', stats.rows_processed > 0)
record(SUITE, 'sorted_stats_rows_merged', stats.rows_merged, 'rows', stats.rows_merged == 500)

# --- merge_stream basic ---
print('\n--- merge_stream (basic) ---')
stats2 = StreamStats()
batches2 = list(merge_stream(iter(a), iter(b), key='id', timestamp_col='ts', stats=stats2))
total2 = sum(len(batch) for batch in batches2)
record(SUITE, 'unsorted_basic_rows', total2, 'rows', total2 == 1500)
record(SUITE, 'unsorted_stats_rows_merged', stats2.rows_merged, 'rows', stats2.rows_merged == 500)

# --- Batch size control ---
print('\n--- Batch size control ---')
stats3 = StreamStats()
batches3 = list(merge_sorted_stream(iter(a), iter(b), key='id', batch_size=200, timestamp_col='ts', stats=stats3))
record(SUITE, 'batch_control_batches', len(batches3), 'batches', len(batches3) > 1)
record(SUITE, 'batch_control_peak', stats3.peak_batch_size, 'rows')

# --- Schema-driven streaming ---
print('\n--- Schema-driven streaming ---')
from crdt_merge import MergeSchema, MaxWins, LWW
schema = MergeSchema(default=LWW(), value=MaxWins())
stats4 = StreamStats()
batches4 = list(merge_sorted_stream(iter(a), iter(b), key='id', schema=schema, timestamp_col='ts', stats=stats4))
total4 = sum(len(batch) for batch in batches4)
record(SUITE, 'schema_stream_rows', total4, 'rows', total4 == 1500)

# --- Scale: 100K records ---
print('\n--- Scale: 100K records ---')
big_a = gen_records(100000, 0)
big_b = gen_records(100000, 50000)
stats5 = StreamStats()
t0 = time.time()
batches5 = list(merge_sorted_stream(iter(big_a), iter(big_b), key='id', timestamp_col='ts', stats=stats5))
elapsed = (time.time() - t0) * 1000
total5 = sum(len(batch) for batch in batches5)
record(SUITE, 'scale_100k_rows', total5, 'rows', total5 == 150000)
record(SUITE, 'scale_100k_time_ms', round(elapsed, 1), 'ms')
record(SUITE, 'scale_100k_rows_per_sec', round(stats5.rows_per_sec, 0), 'rows/s')

# --- Memory measurement ---
print('\n--- Memory: merge_sorted_stream 100K ---')
def stream_100k():
    aa = gen_records(100000, 0)
    bb = gen_records(100000, 50000)
    total = 0
    for batch in merge_sorted_stream(iter(aa), iter(bb), key='id', timestamp_col='ts'):
        total += len(batch)
    return total

rows, peak_mb = measure_memory(stream_100k)
record(SUITE, 'sorted_100k_peak_mb', round(peak_mb, 1), 'MB')
record(SUITE, 'sorted_100k_rows', rows, 'rows', rows == 150000)

print(f'\nStreaming suite: {sum(1 for r in results if r["suite"]==SUITE and r["passed"])}/{sum(1 for r in results if r["suite"]==SUITE)} passed')

## Suite 4: System Sanity Check
Quick validation that all v0.3.0 core features still work correctly.

In [ ]:
from crdt_merge import merge, diff, GCounter, PNCounter, LWWRegister, ORSet, LWWMap
from crdt_merge import MergeSchema, LWW, MaxWins, MinWins, Concat
from crdt_merge.delta import compute_delta, apply_delta, compose_deltas, DeltaStore

print('=' * 70)
print('SUITE 4: SYSTEM SANITY')
print('=' * 70)

SUITE = 'sanity'

# --- Core merge ---
print('\n--- Core merge ---')
a = [{'id': 1, 'v': 'a', 'ts': 1.0}, {'id': 2, 'v': 'a', 'ts': 1.0}]
b = [{'id': 2, 'v': 'b', 'ts': 2.0}, {'id': 3, 'v': 'b', 'ts': 1.0}]
merged = merge(a, b, key='id', timestamp_col='ts')
record(SUITE, 'core_merge', len(merged), 'rows', len(merged) == 3)
id2 = [r for r in merged if r['id'] == 2][0]
record(SUITE, 'lww_correct', id2['v'], '', id2['v'] == 'b')

# --- Diff ---
print('\n--- Diff ---')
d = diff(a, b, key='id')
record(SUITE, 'diff_works', len(str(d)), 'chars', isinstance(d, dict))

# --- CRDTs ---
print('\n--- CRDTs ---')
gc1 = GCounter(); gc2 = GCounter()
gc1.increment('node_a', 5); gc2.increment('node_b', 3)
gc3 = gc1.merge(gc2)
record(SUITE, 'gcounter_merge', gc3.value, '', gc3.value == 8)

pn1 = PNCounter(); pn2 = PNCounter()
pn1.increment('a', 10); pn2.decrement('b', 3)
pn3 = pn1.merge(pn2)
record(SUITE, 'pncounter_merge', pn3.value, '', pn3.value == 7)

r1 = LWWRegister('hello', timestamp=1.0)
r2 = LWWRegister('world', timestamp=2.0)
r3 = r1.merge(r2)
record(SUITE, 'lww_register', r3.value, '', r3.value == 'world')

s1 = ORSet(); s2 = ORSet()
s1.add('a'); s1.add('b'); s2.add('b'); s2.add('c')
s3 = s1.merge(s2)
record(SUITE, 'orset_merge', len(s3.value), 'elements', len(s3.value) == 3)

m1 = LWWMap(); m2 = LWWMap()
m1.set('key', 'val1', timestamp=1.0); m2.set('key', 'val2', timestamp=2.0)
m3 = m1.merge(m2)
record(SUITE, 'lwwmap_merge', m3.get('key'), '', m3.get('key') == 'val2')

# --- Strategies ---
print('\n--- Strategies ---')
schema = MergeSchema(default=LWW(), v=MaxWins())
a = [{'id': 1, 'v': 10, 'ts': 1.0}]
b = [{'id': 1, 'v': 20, 'ts': 0.5}]
m = merge(a, b, key='id', timestamp_col='ts')
# Note: schema not passed to merge() directly, testing strategies work:
m2, _ = merge_with_provenance(a, b, key='id', schema=schema, timestamp_col='ts')
id1 = [r for r in m2 if r['id'] == 1][0]
record(SUITE, 'maxwins_strategy', id1['v'], '', id1['v'] == 20)

# --- Delta ---
print('\n--- Delta ---')
old = [{'id': 1, 'v': 'old'}, {'id': 2, 'v': 'keep'}]
new = [{'id': 1, 'v': 'new'}, {'id': 2, 'v': 'keep'}, {'id': 3, 'v': 'added'}]
delta = compute_delta(old, new, key='id')
record(SUITE, 'delta_compute', delta.size, 'changes', delta.size > 0)
applied = apply_delta(old, delta, key='id')
record(SUITE, 'delta_apply', len(applied), 'rows', len(applied) == 3)

# --- Scale: 50K merge ---
print('\n--- Scale: 50K merge ---')
a_big = gen_records(50000, 0)
b_big = gen_records(50000, 25000)
t0 = time.time()
merged_big = merge(a_big, b_big, key='id', timestamp_col='ts')
elapsed = (time.time() - t0) * 1000
record(SUITE, 'scale_50k_rows', len(merged_big), 'rows', len(merged_big) == 75000)
record(SUITE, 'scale_50k_time_ms', round(elapsed, 1), 'ms')

print(f'\nSanity suite: {sum(1 for r in results if r["suite"]==SUITE and r["passed"])}/{sum(1 for r in results if r["suite"]==SUITE)} passed')

## Final Report

In [ ]:
print('=' * 70)
print('FINAL REPORT: crdt-merge v0.4.0 A100 Validation')
print('=' * 70)

total = len(results)
passed = sum(1 for r in results if r['passed'])
failed = total - passed

print(f'\nTotal measurements: {total}')
print(f'Passed: {passed}')
print(f'Failed: {failed}')

for suite in ['provenance', 'verify', 'streaming', 'sanity']:
    suite_results = [r for r in results if r['suite'] == suite]
    sp = sum(1 for r in suite_results if r['passed'])
    st = len(suite_results)
    status = '' if sp == st else ''
    print(f'  {status} {suite}: {sp}/{st}')

if failed > 0:
    print('\n FAILURES:')
    for r in results:
        if not r['passed']:
            print(f'  - {r["suite"]}/{r["test"]}: {r["value"]} {r["unit"]}')

print(f'\n{" ALL TESTS PASSED" if failed == 0 else "️ SOME TESTS FAILED"}')

# Export results
with open('v040_a100_results.json', 'w') as f:
    json.dump({'version': '0.4.0', 'measurements': results,
               'total': total, 'passed': passed, 'failed': failed}, f, indent=2)
print(f'\nResults exported to v040_a100_results.json')